# Production-Grade Ensemble CNN Classifier with Performance Benchmarks

**Task 15**

## 1. Project Overview

In this mini project, we build a production-grade image classification system using multiple Convolutional Neural Network (CNN) models.

Instead of training only one CNN and using its prediction, we train **multiple CNN models** and combine their predictions to produce one final result. This approach is called **Ensemble Learning**.

The project focuses on two important areas:
1. Improving classification quality using multiple CNN models.
2. Measuring whether the improvement is worth the additional production cost.

We compare:
- Individual CNN performance (baseline, deep, regularized)
- Ensemble CNN performance (soft-voting average of predictions)
- Robustness under noisy / perturbed inputs
- Training/inference cost vs. accuracy gain trade-off

## 2. Setup

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODELS_DIR = "../models"
IMAGES_DIR = "../report"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

## 3. Load and Prepare Data

Using CIFAR-10 as the benchmark dataset (swap out for your own dataset if needed).

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

num_classes = 10
input_shape = x_train.shape[1:]
print("Train:", x_train.shape, "Test:", x_test.shape)

## 4. Define Three CNN Architectures

- **cnn_baseline** — a simple, shallow CNN (fast, lower capacity)
- **cnn_deep** — a deeper CNN with more conv blocks (higher capacity)
- **cnn_regularized** — same depth as baseline but with Dropout + BatchNorm + L2 (better generalization)

In [ ]:
def build_cnn_baseline():
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_baseline")
    return model

def build_cnn_deep():
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_deep")
    return model

def build_cnn_regularized():
    reg = keras.regularizers.l2(1e-4)
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, padding="same", kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding="same", kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(128, activation="relu", kernel_regularizer=reg),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_regularized")
    return model

builders = {
    "cnn_baseline": build_cnn_baseline,
    "cnn_deep": build_cnn_deep,
    "cnn_regularized": build_cnn_regularized,
}

## 5. Train All Three Models

In [ ]:
EPOCHS = 15
BATCH_SIZE = 64

histories = {}
train_times = {}
models = {}

for name, builder in builders.items():
    print(f"\n=== Training {name} ===")
    model = builder()
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    start = time.time()
    history = model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2,
    )
    elapsed = time.time() - start
    train_times[name] = elapsed
    histories[name] = history.history
    models[name] = model
    model.save(os.path.join(MODELS_DIR, f"{name}.keras"))
    print(f"Saved {name}.keras — trained in {elapsed:.1f}s")

## 6. Individual Model Performance

In [ ]:
individual_results = {}
for name, model in models.items():
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    individual_results[name] = {"loss": loss, "accuracy": acc}
    print(f"{name}: accuracy={acc:.4f}, loss={loss:.4f}")

In [ ]:
# cnn_independent.png — bar chart comparing standalone model accuracy
plt.figure(figsize=(6, 4))
names = list(individual_results.keys())
accs = [individual_results[n]["accuracy"] for n in names]
plt.bar(names, accs, color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("Test Accuracy")
plt.title("Independent CNN Performance")
plt.ylim(0, 1)
for i, a in enumerate(accs):
    plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "cnn_independent.png"), dpi=150)
plt.show()

## 7. Ensemble (Soft Voting) Performance

In [ ]:
def ensemble_predict(models_dict, x):
    preds = [m.predict(x, verbose=0) for m in models_dict.values()]
    avg_preds = np.mean(preds, axis=0)
    return avg_preds

ensemble_probs = ensemble_predict(models, x_test)
ensemble_pred_labels = np.argmax(ensemble_probs, axis=1)
ensemble_accuracy = np.mean(ensemble_pred_labels == y_test)
print(f"Ensemble accuracy: {ensemble_accuracy:.4f}")

In [ ]:
# cnn_ensemble.png — comparison of individual models vs the ensemble
plt.figure(figsize=(6, 4))
all_names = names + ["ensemble"]
all_accs = accs + [ensemble_accuracy]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
plt.bar(all_names, all_accs, color=colors)
plt.ylabel("Test Accuracy")
plt.title("Individual Models vs Ensemble")
plt.ylim(0, 1)
for i, a in enumerate(all_accs):
    plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "cnn_ensemble.png"), dpi=150)
plt.show()

## 8. Training Curves (Accuracy / Loss)

In [ ]:
# acc_loss.png — training/validation accuracy and loss curves for all models
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, hist in histories.items():
    axes[0].plot(hist["accuracy"], label=f"{name} (train)")
    axes[0].plot(hist["val_accuracy"], linestyle="--", label=f"{name} (val)")
    axes[1].plot(hist["loss"], label=f"{name} (train)")
    axes[1].plot(hist["val_loss"], linestyle="--", label=f"{name} (val)")

axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend(fontsize=7)
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "acc_loss.png"), dpi=150)
plt.show()

## 9. Robustness Test (Noisy Inputs)

Simulates production conditions by adding Gaussian noise to test images and measuring accuracy degradation for each model vs. the ensemble.

In [ ]:
def add_noise(x, sigma):
    noisy = x + np.random.normal(0, sigma, x.shape)
    return np.clip(noisy, 0.0, 1.0).astype("float32")

noise_levels = [0.0, 0.05, 0.1, 0.2, 0.3]
robustness = {name: [] for name in models}
robustness["ensemble"] = []

for sigma in noise_levels:
    x_noisy = add_noise(x_test, sigma)
    preds_all = []
    for name, model in models.items():
        p = model.predict(x_noisy, verbose=0)
        preds_all.append(p)
        acc = np.mean(np.argmax(p, axis=1) == y_test)
        robustness[name].append(acc)
    ens_p = np.mean(preds_all, axis=0)
    ens_acc = np.mean(np.argmax(ens_p, axis=1) == y_test)
    robustness["ensemble"].append(ens_acc)
    print(f"sigma={sigma}: " + ", ".join(f"{k}={v[-1]:.3f}" for k, v in robustness.items()))

In [ ]:
# robustness.png — accuracy vs noise level for each model + ensemble
plt.figure(figsize=(6.5, 4.5))
for name, accs_by_noise in robustness.items():
    style = "-o" if name == "ensemble" else "--o"
    lw = 2.5 if name == "ensemble" else 1.5
    plt.plot(noise_levels, accs_by_noise, style, linewidth=lw, label=name)
plt.xlabel("Gaussian noise sigma")
plt.ylabel("Test Accuracy")
plt.title("Robustness Under Noisy Inputs")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "robustness.png"), dpi=150)
plt.show()

## 10. Cost vs. Benefit Summary

Is the ensemble worth the extra production cost (3x inference calls, 3x model storage, 3x training time)?

In [ ]:
import pandas as pd

summary_rows = []
for name in names:
    summary_rows.append({
        "model": name,
        "test_accuracy": individual_results[name]["accuracy"],
        "train_time_sec": train_times[name],
        "params": models[name].count_params(),
    })
summary_rows.append({
    "model": "ensemble (all 3)",
    "test_accuracy": ensemble_accuracy,
    "train_time_sec": sum(train_times.values()),
    "params": sum(models[n].count_params() for n in names),
})

summary_df = pd.DataFrame(summary_rows)
summary_df["accuracy_gain_vs_best_single"] = summary_df["test_accuracy"] - max(accs)
summary_df.to_csv(os.path.join(IMAGES_DIR, "summary_metrics.csv"), index=False)
summary_df

## 11. Conclusion

Fill in after running:
- Best single model: ______
- Ensemble accuracy gain: ______ percentage points
- Ensemble robustness gain under noise: ______
- Extra cost (compute/storage/latency): ______
- Recommendation (ensemble vs single model for production): ______